In [3]:
import requests

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def send_request():
    r = requests.post(url)
    print(r.status_code)
    return r.status_code
    
    
with ThreadPoolExecutor(max_workers=50) as executor:
    futures = [executor.submit(send_request) for r in range(0, 10000)]


In [3]:
def get_problem(url):
    r = requests.get(url)
    return r.text

In [4]:
import requests
from datetime import datetime, timezone

def send_solution(id, code, cuname):
    url = ('https://codingbat.com/run')
    
    headers = {
        "Accept": "*/*",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "Accept-Language": "en-US,en;q=0.9",

        "Content-Type": "application/x-www-form-urlencoded",
        "Origin": "https://codingbat.com",
        "Referer": "https://codingbat.com/prob/p181646",
        "Priority": "u=0",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
        "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:151.0) Gecko/20100101 Firefox/151.0",
    }

    cookies = {
        "JSESSIONID": "F48E2FA8DD4DCD297463058A17D5AA1C"
    }

    data = {
        "id": id,
        "code": code,
        "cuname": cuname,
        "date":int(datetime.now(timezone.utc).timestamp()),
        "adate": datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%Sz").lower()
    }

    r = requests.post(
        url,
        headers=headers,
        cookies=cookies,
        data=data,
    )

    return r

text = send_solution("p154485", "def hello():\n    return 'Hello, World!'", "jayverma123@gmail.com")

In [5]:
import ollama

def prompt(problem):
    return f"""
    You are an expert Java developer solving CodingBat exercises.

    Problem:
    {problem}

    Write the Java method to solve this problem. 
    CRITICAL INSTRUCTIONS:
    - Output ONLY the raw, valid Java code.
    - Do not include any explanations, greetings, or markdown formatting (like ```java).
    - Do not wrap the method in a class definition.
    - The output must be immediately ready to copy-paste into the CodingBat editor to compile and run.
    """

def ollama_nemtron_3_request(prompt):
    response = ollama.chat(
        model='nemotron-3-nano:30b-cloud',
        messages=[
            {
                'role': 'user',
                'content': prompt
            }
        ]
    )


    return (response.message.content)


In [14]:
from bs4 import BeautifulSoup
from pathlib import Path


def extract_next_link(html):
    soup = BeautifulSoup(html, 'html.parser')

    next_button = soup.find('a', string='next')
    if next_button:
        next_id = next_button.get('href')
        if next_id.startswith("/prob/"):
            next_id = next_id[len("/prob/"):]
            
        return next_id
    else:
        return None


def get_codingbat_problem(html):
    soup  = BeautifulSoup(html, 'html.parser')
    problem = soup.find('p', class_='max2')
    if problem:
        problem_text = problem.get_text(strip=True)
        return problem_text
    else:
        raise ValueError("No problem found")


def extract_coding_bat_problem_title(html):
    soup = BeautifulSoup(html, 'html.parser')
    title_element = soup.find_all('span', class_='h2')[1]
    if title_element:
        return title_element.get_text(strip=True)
    else:
        raise ValueError("No title found")
    

def get_solution_by_title(problem_title, answers_path="/home/jayv/Documents/CodingBat.Com-Agent/answers"):
    root = Path(answers_path)


    for file_path in root.rglob("*"):
        if file_path.is_file() and file_path.stem == problem_title:
            return file_path.read_text(encoding="utf-8")

    return None

In [15]:
base_url = "https://codingbat.com/prob/"
next_id = "p187868"


while next_id:
    page_content = get_problem(base_url + next_id)
    
    print(get_solution_by_title(extract_coding_bat_problem_title(page_content)))
    break
    
    
    print(base_url + next_id)
    
    problem = get_codingbat_problem(page_content)
    prompt_text = prompt(problem)
    solution = ollama_nemtron_3_request(prompt_text)
    status = send_solution(id=next_id, code=solution, cuname="jayverma123@gmail.com")
    next_id = extract_next_link(page_content)
    
 

/* The parameter weekday is true if it is a weekday, and the parameter 
 * vacation is true if we are on vacation. We sleep in if it is not a weekday 
 * or we're on vacation. Return true if we sleep in.
 */
public boolean sleepIn(boolean weekday, boolean vacation) {
    return !weekday || vacation;
}



In [3]:
from pathlib import Path
import shutil


source_root = Path("/home/jayv/Documents/CodingBat.Com-Agent/coding_bat_answers_repo")
dump_folder = Path("/home/jayv/Documents/CodingBat.Com-Agent/answers")

for file_path in source_root.rglob("*"):
    if file_path.is_file():
        dest = dump_folder / file_path.name
        
        shutil.move(str(file_path), str(dest))